# EXP-001 — Train thử 1 seed với trọng số AU (`img-templ-sel-smile`)
Notebook độc lập cho runtime **A100** mới. Các bước:
1. Tính trước trên **toàn bộ** CAER-S crop mặt (~49K ảnh): vector target → câu mẫu AU → py-feat (**lâu nhất, ~4 giờ**) → trọng số.
2. Train seed 1 với `--sample-weights` (5 epoch, ~10 phút/epoch).
3. Test zero-shot, so với EXP-000 seed 1, EXP-000 trung bình 3 seed và paper.

Mọi stage **chạy tiếp được** nếu runtime bị ngắt: kết quả lưu dần lên Drive (`ExpCLIP_output/exp001/full/`). Bị ngắt thì chạy lại notebook từ đầu, các stage sẽ bỏ qua phần đã xong.

# Cấu hình

In [1]:
SEED = 1
JOB_ID = f"exp001_au_l14_seed{SEED}"
BRANCH = "exp/001-au-detector-filtering"

# trọng số: cấu hình chọn ở test04
POOL, PROTO, MAP, BALANCE, W_MIN = "img", "templ", "linear", "smile", 0.1
GROUPS_SEL = ["upper_lid_raise", "cheek_raise", "lip_corner_pull", "dimpler",
              "lip_stretch", "mouth_open", "eyes_closed"]   # Spearman > 0.1 ở test03
WEIGHTS_FILE = f"au_weights_{POOL}-{PROTO}-sel-{BALANCE}.json"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"     # precompute toàn bộ ảnh (dùng chung cho các seed)
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/train"    # checkpoint, log, kết quả test
T5_CACHE = "/root/.cache/huggingface/hub/models--google--flan-t5-xxl"
SCRIPT = "tools/precompute_au_weights.py"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

def hf_offline():
    # FlanT5 đã có trong cache thì không gọi HuggingFace nữa
    os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

# Mount Drive & GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(PRE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
# BLIP-2 FlanT5-XXL cần ~24GB chỉ để nạp model → GPU 22–24GB (L4, A10) sẽ hết bộ nhớ
gpu_mem = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).split()[0])
assert gpu_mem >= 35000, f"GPU chỉ có {gpu_mem} MiB — đổi runtime sang A100 (Runtime > Change runtime type)"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


# Clone repo (nhánh EXP-001)

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

/content/Exp-CLIP
M	confusion_matrix/CLIP_L14-AffectNet7-type1.pdf
M	confusion_matrix/CLIP_L14-AffectNet7-type2.pdf
M	confusion_matrix/CLIP_L14-AffectNet7-type3.pdf
M	confusion_matrix/CLIP_L14-AffectNet7-type4.pdf
M	confusion_matrix/CLIP_L14-AffectNet7-type5.pdf
M	confusion_matrix/CLIP_L14-AffectNet8-type1.pdf
M	confusion_matrix/CLIP_L14-AffectNet8-type2.pdf
M	confusion_matrix/CLIP_L14-AffectNet8-type3.pdf
M	confusion_matrix/CLIP_L14-AffectNet8-type4.pdf
M	confusion_matrix/CLIP_L14-AffectNet8-type5.pdf
M	confusion_matrix/CLIP_L14-FERPlus-type1.pdf
M	confusion_matrix/CLIP_L14-FERPlus-type2.pdf
M	confusion_matrix/CLIP_L14-FERPlus-type3.pdf
M	confusion_matrix/CLIP_L14-FERPlus-type4.pdf
M	confusion_matrix/CLIP_L14-FERPlus-type5.pdf
M	confusion_matrix/CLIP_L14-RAFDB-type1.pdf
M	confusion_matrix/CLIP_L14-RAFDB-type2.pdf
M	confusion_matrix/CLIP_L14-RAFDB-type3.pdf
M	confusion_matrix/CLIP_L14-RAFDB-type4.pdf
M	confusion_matrix/CLIP_L14-RAFDB-type5.pdf
M	models/clip/__pycache__/__init__.cpython

# Giải nén dữ liệu (train + test)

In [4]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [5]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
salesforce-lavis 1.0.2 requires opencv-python-headless==4.5.5.64, but you have opencv-python-headless 4.10.0.84 which is incompatible.
numpy 1.26.4 | torch 2.0.1+cu118 | cuda True


# Precompute 1 — Vector target (toàn bộ ảnh)

In [6]:
hf_offline()
run(f'/content/venv310/bin/python {SCRIPT} target --train-data {TRAIN_DATA} --out-dir {PRE_DIR} '
    f'--limit 0 --batch-size 64 --workers 8 --instruction "{INSTRUCTION}"')

0 done, 48938 to encode
/content/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

  0%|          | 0.00/1.89G [00:00<?, ?B/s]
  0%|          | 48.0k/1.89G [00:00<2:43:43, 206kB/s]
  0%|          | 80.0k/1.89G [00:00<3:08:32, 179kB/s]
  0%|          | 144k/1.89G [00:00<1:48:54, 310kB/s] 
  0%|          | 192k/1.89G [00:00<1:36:58, 348kB/s]
  0%|          | 272k/1.89G [00:00<1:13:06, 462kB/s]
  0%|          | 400k/1.89G [00:00<49:48, 677kB/s]  
  0%|          | 544k/1.89G [00:00<38:43, 871kB/s]
  0%|          | 784k/1.89G [00:01<25:55, 1.30MB/s]
  0%|          | 1.05M/1.89G [00:01<19:06, 1.76MB/s]
  0%|          | 1.48M/1.89G [00:01<13:11, 2.56MB/s]
  0%|          | 2.12M/1.89G [00:01<09:03, 3.72MB/s]
  0%|          | 2.91M/1.89G [00:01<06:39, 5.06MB/s]
  0%| 

# Precompute 2 — Vector câu mẫu AU

In [7]:
hf_offline()
if not os.path.exists(f"{PRE_DIR}/protos.npz"):
    run(f'/content/venv310/bin/python {SCRIPT} proto --out-dir {PRE_DIR} --instruction "{INSTRUCTION}"')
!ls -lh "$PRE_DIR/protos.npz"; ls "$PRE_DIR/targets" | wc -l

/content/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

Loading checkpoint shards: 100%|██████████| 5/5 [02:01<00:00, 24.34s/it]
plain: mean cosine between different groups 0.845
templ: mean cosine between different groups 0.970
saved 13 prototypes
-rw------- 1 root root 418K Oct 13 15:29 /content/drive/MyDrive/ExpCLIP_output/exp001/full/protos.npz
765


# Cài py-feat (venv_feat)\nNếu `aus.csv` trên Drive đã đủ mọi ảnh (py-feat chạy xong ở lần trước) thì **bỏ qua** cài đặt và stage py-feat. Giữ cache FlanT5 để lát train không phải tải lại; chỉ xoá khi ổ đĩa gần đầy.

In [8]:
import csv, glob
n_images = len(glob.glob(f"{TRAIN_DATA}/*/*"))
n_aus = 0
if os.path.exists(f"{PRE_DIR}/aus.csv"):
    with open(f"{PRE_DIR}/aus.csv") as f:
        n_aus = sum(1 for _ in csv.DictReader(f))
AU_DONE = n_aus >= n_images
print(f"aus.csv: {n_aus}/{n_images} ảnh →", "đã xong, bỏ qua py-feat" if AU_DONE else "cần chạy py-feat")

aus.csv: 0/48938 ảnh → cần chạy py-feat


In [9]:
!rm -rf /root/.cache/pip
free_gb = int(subprocess.check_output("df --output=avail -BG /content | tail -1", shell=True).decode().strip().rstrip("G"))
print("Ổ đĩa còn trống:", free_gb, "GB")
if free_gb < 25:
    print("→ xoá cache FlanT5 (train sẽ tải lại)")
    !rm -rf /root/.cache/huggingface

Ổ đĩa còn trống: 139 GB


In [10]:
if not AU_DONE:
    !rm -rf /content/venv_feat   # xoá bản cài dở nếu có
    !python3.10 -m venv /content/venv_feat
    !/content/venv_feat/bin/pip install -q --upgrade pip
    # py-feat 0.6.x cần bản cũ (cài chung 1 lệnh để pip không tự nâng):
    #   torchvision mới đã bỏ `torchvision.io.read_video`; scipy 1.14 bỏ `simps`; chưa hỗ trợ numpy 2
    !/content/venv_feat/bin/pip install -q --no-cache-dir py-feat==0.6.2 torch==2.2.2 torchvision==0.17.2 "numpy<2" "scipy<1.14" --extra-index-url https://download.pytorch.org/whl/cu121
    !MPLBACKEND=Agg /content/venv_feat/bin/python -c "import feat, torch, torchvision; from feat import Detector; print('py-feat', feat.__version__, '| torchvision', torchvision.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.8 MB/s eta 0:00:00
py-feat 0.6.2 | torchvision 0.17.2+cu121 | torch 2.2.2+cu121 | cuda True


# Precompute 3 — py-feat (toàn bộ ảnh, ~4 giờ)\nGhi dần vào `aus.csv` trên Drive; bị ngắt thì chạy lại cell này, nó làm tiếp phần còn thiếu.

In [11]:
if not AU_DONE:
    # MPLBACKEND=Agg: venv_feat không có backend inline của notebook, matplotlib sẽ báo lỗi khi import feat
    run(f'MPLBACKEND=Agg /content/venv_feat/bin/python {SCRIPT} au --train-data {TRAIN_DATA} --out-dir {PRE_DIR} '
        f'--limit 0 --batch-size 16')

Streaming output truncated to the last 5000 lines.
33984/48938

100%|██████████| 1/1 [00:04<00:00,  4.18s/it]
34000/48938

100%|██████████| 1/1 [00:04<00:00,  4.38s/it]
34016/48938

  0%|          | 0/1 [00:00<?, ?it/s]/content/venv_feat/lib/python3.10/site-packages/feat/detector.py:1143: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(out)

100%|██████████| 1/1 [00:03<00:00,  3.95s/it]
34032/48938

100%|██████████| 1/1 [00:04<00:00,  4.31s/it]
34048/48938

  0%|          | 0/1 [00:00<?, ?it/s]/content/venv_feat/lib/python3.10/site-packages/feat/detector.py:1143: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA co

# Precompute 4 — Trọng số

In [12]:
run(f'/content/venv310/bin/python {SCRIPT} weight --out-dir {PRE_DIR} --pool {POOL} --proto {PROTO} --map {MAP} '
    f'--w-min {W_MIN} --groups {",".join(GROUPS_SEL)} --balance {BALANCE}')
!ls -lh "$PRE_DIR/$WEIGHTS_FILE"

[img-templ-sel-smile] 48938 targets | 48328 with AUs | 610 no face (w=1.0) | 0 not yet AU-detected (skipped)
agreement over 7 groups: mean 0.214 std 0.412 | weight mean 0.646 std 0.184 (map linear, balance smile, w_min 0.1)
spearman per group: brow_raise -0.14, brow_lower +0.05, upper_lid_raise +0.20, cheek_raise +0.10, lid_tight -0.16, nose_wrinkle +0.02, lip_corner_pull +0.39, dimpler +0.14, lip_corner_depress -0.06, lip_stretch +0.23, lip_press +0.01, mouth_open +0.22, eyes_closed +0.32
-rw------- 1 root root 1.3M Oct 13 21:36 /content/drive/MyDrive/ExpCLIP_output/exp001/full/au_weights_img-templ-sel-smile.json


# Train seed 1 với trọng số\nĐúng cấu hình EXP-000 (runner.sh), chỉ thêm `--sample-weights`. Checkpoint/log copy lên `OUT_DIR` sau mỗi epoch.

In [13]:
hf_offline()
resume = f"{OUT_DIR}/{JOB_ID}-resume.pth"
run(f'/content/venv310/bin/python train.py '
    f'--workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
    f'--lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed {SEED} '
    f'--job-id {JOB_ID} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
    f'--train-data {TRAIN_DATA} --extra-checkpoint-dir {OUT_DIR} '
    f'--sample-weights {PRE_DIR}/{WEIGHTS_FILE}'
    + (f' --resume {resume}' if os.path.exists(resume) else ''))

Training date:  26-10-13 21:36
************************
workers = 8
epochs = 5
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 10
milestones = [30]
seed = 1
job_id = exp001_au_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp001/train
train_data = /content/CAER-S_Face/train
sample_weights = /content/drive/MyDrive/ExpCLIP_output/exp001/full/au_weights_img-templ-sel-smile.json
resume = None
************************

  0%|                                               | 0.00/890M [00:00<?, ?iB/s]
  3%|█                                      | 23.5M/890M [00:00<00:03, 234MiB/s]
  5%|█▉                                    | 45.8M/890M [00:00<00:10, 88.2MiB/s]
  7%|██▍                                   | 58.4M/890M [00:04<01

# Test zero-shot

In [6]:
!mkdir -p checkpoint log
if not os.path.exists(f"checkpoint/{JOB_ID}-model.pth"):
    !cp "$OUT_DIR/$JOB_ID-model.pth" checkpoint/
run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {JOB_ID} '
    f'--data-root /content/static_test_sets 2>&1 | tee log/{JOB_ID}-test.txt')
!cp log/$JOB_ID-test.txt "$OUT_DIR/"

************************
load_model = CLIP_L14
job_id = exp001_au_l14_seed1
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 50.21/56.55
************************* AffectNet7
UAR/WAR: 42.60/42.60
************************* AffectNet8
UAR/WAR: 36.95/36.95
************************* FERPlus
UAR/WAR: 38.19/45.40
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 56.59/63.92
************************* AffectNet7
UAR/WAR: 43.97/43.97
************************* AffectNet8
UAR/WAR: 38.33/38.33
************************* FERPlus
UAR/WAR: 39.22/52.70
********************************

# So sánh với EXP-000 và paper\nΔ = EXP-001 − mốc so sánh. Lưu ý: 1 seed; EXP-000 giữa các seed lệch ~0.5–1.8 điểm (RAFDB WAR tới 2.9), nên Δ nhỏ hơn mức đó chưa nói lên gì.

In [7]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

rows, prompt, dataset = [], None, None
for line in open(f"log/{JOB_ID}-test.txt"):
    if "Prompt Type:" in line:
        prompt = line.split("Prompt Type:")[1].strip()
    elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
        dataset = line.strip().split()[-1]
    elif line.startswith("UAR/WAR:"):
        uar, war = map(float, line.split(":")[1].strip().split("/"))
        rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/{JOB_ID}-results.csv", index=False)

table = res.set_index(["prompt", "dataset"])
for src in ["EXP-000 seed1", "EXP-000 mean3", "paper"]:
    ref = REF[REF.src == src].set_index(["prompt", "dataset"])
    table[f"ΔUAR vs {src}"] = (table.UAR - ref.UAR).round(2)
table[f"ΔWAR vs EXP-000 seed1"] = (table.WAR - REF[REF.src == "EXP-000 seed1"].set_index(["prompt", "dataset"]).WAR).round(2)
pd.set_option("display.width", 250)
table

UAR    WAR  ΔUAR vs EXP-000 seed1  ΔUAR vs EXP-000 mean3  ΔUAR vs paper  ΔWAR vs EXP-000 seed1
prompt                                       dataset                                                                                                     
Class Name                                   RAFDB       50.21  56.55                  -0.15                  -0.02          -8.49                  -0.03
                                             AffectNet7  42.60  42.60                   0.09                   1.13          -1.67                   0.09
                                             AffectNet8  36.95  36.95                   0.00                   1.19          -1.49                   0.00
                                             FERPlus     38.19  45.40                  -0.08                  -0.39         -10.09                  -0.05
An Expression of Name                        RAFDB       56.59  63.92                  -0.14                   1.25          -2.11                  -0.13
                                             AffectNet7  43.97  43.97                  -0.03                   0.18          -0.30                  -0.03
                                             AffectNet8  38.33  38.33                   0.00                  -0.07          -0.11                   0.00
                                             FERPlus     39.22  52.70                  -0.77                   0.33          -9.06                  -0.22
A Photo of A Face with An Expression of Name RAFDB       54.88  65.09                  -0.06                  -0.76          -3.82                  -0.03
                                             AffectNet7  44.03  44.03                   0.06                  -0.36          -0.24                   0.06
                                             AffectNet8  38.05  38.05                   0.03                  -0.03          -0.39                   0.03
                                             FERPlus     36.70  49.01                   0.07                  -1.25         -11.58                   0.12
Expression Ensemble Five                     RAFDB       58.47  66.79                  -0.41                   0.40          -0.23                  -0.09
                                             AffectNet7  44.06  44.06                  -0.03                  -0.11          -0.21                  -0.03
                                             AffectNet8  39.05  39.05                  -0.18                   0.21           0.61                  -0.18
                                             FERPlus     40.61  51.47                  -0.04                  -0.94          -7.67                   0.00
Expression Ensemble Ten                      RAFDB       57.84  66.00                   0.11                   0.65          -0.86                   0.09
                                             AffectNet7  44.94  44.94                   0.00                  -0.03           0.67                   0.00
                                             AffectNet8  39.60  39.60                   0.05                   0.15           1.16                   0.05
                                             FERPlus     40.55  49.40                   0.10                  -1.62          -7.73                   0.11

# Kiểm tra log train\nSo loss/accuracy từng epoch với EXP-000 (loss có trọng số nên giá trị tuyệt đối không so trực tiếp được, chỉ xem có hội tụ bình thường không).

In [17]:
!grep -E "Sample weights|The best accuracy|An epoch time" log/$JOB_ID-log.txt

Sample weights: 48938 / 48938 images found in /content/drive/MyDrive/ExpCLIP_output/exp001/full/au_weights_img-templ-sel-smile.json (others = 1), mean 0.651, min 0.105
The best accuracy: 1.3692435026168823
An epoch time: 511.4391634464264s
The best accuracy: 5.462582588195801
An epoch time: 503.61780190467834s
The best accuracy: 12.600740432739258
An epoch time: 504.77510380744934s
The best accuracy: 21.007402420043945
An epoch time: 504.55810832977295s
The best accuracy: 27.611021041870117
An epoch time: 505.2109935283661s
